### TASK 3 : Hands on/Homework

Repeat the above exercises for another region that interests you, (e.g. West African monsoon, south American monsoon, Europe, take an area that interests you). Remember that if you cut down the seasons within the year to target the months when the rains arrive if you are focusing on a monsoon region!


### Area of Interest

The area of Interest is the Philippine Area of Responsibility (PAR) which spans $115^\circ-135^\circ$ East and $5^\circ-25^\circ$ North. We are analyzing the precipitation in $mm/day$ from 1997 to 2021.

### 1. Annual Precipitation Anomaly (mm/day)
<img src="PHanomaly.gif" alt="PHanomaly" width="fit"/>

### 2. Mean Annual Precipitation Anomaly (mm/day)
<img src="1996_2021_PH_Anomaly.png" alt="1996_2021_PH_Anomaly" width="fit"/>

### 3. Annual Wet Area Index (Fraction of Positive Anomaly)
<img src="1996_2021_PH_WetIndex.png" alt="1996_2021_PH_WetIndex" width="fit"/>

### 4. Annual 95th Percentile of Precipitation
<img src="PHp95.gif" alt="PHp95" width="fit"/>

In [1]:
year1=1997
year2=2021
ddir="/mnt/c/Users/Bea/Downloads/DATA/gpcp"
# has to be an empty folder, or else errors
fname=gpcp_v01r03_daily_

In [2]:
# Getting the data

stub=gpcp_v01r03_daily_
for year in $(seq ${year1} ${year2}); do
   file=${stub}${year}.nc
   if [ ! -f "$FILE_PATH" ]; then
       wget -q -c -P $ddir http://clima-dods.ictp.it/Users/tompkins/Observations/GPCP/v1.3/${stub}${year}.nc
       #wget -q turns off output (quiet)
       #wget -c resume getting a partially-downloaded file
       #wget -P names directory path
   fi
done

In [3]:
# Cutting out the area of interest

# lon1, lon2, lat1, lat2
india="65,90,8,27"
wafrica="-20,20,-5,30"
philippines="115,135,5,25"

region=$philippines
region_tag=_area$(echo $region | tr , _) 

for year in $(seq ${year1} ${year2}); do
#seq makes sequence of numbers, first last, or first increment last
    ifile=${ddir}/${fname}${year}.nc #declaring input file per year
    ofile=${ddir}/${fname}${year}${region_tag}.nc #declaring output file per year
    cdo -s sellonlatbox,${region} $ifile $ofile #sellecting data within the longitude latitude box
    #cdo -s means silent
done

In [4]:
# Merging the files
    
cdo -s -O mergetime ${ddir}/${fname}????${region_tag}.nc ${ddir}/${fname}${region_tag}.nc
#cdo -O means overwrite file if it exists already
#cdo mergetime concatenates files chronologically to make a continuous time series

# Selecting the monsoon season of the Philippines: May to October (5/10)
cdo -s -O selmon,5/10 ${ddir}/${fname}${region_tag}.nc ${ddir}/${fname}${region_tag}_MJJASO.nc
#cdo selmon extracts specific months from a file

#ncdump -h ${ddir}/${fname}${region_tag}_MJJASO.nc
#ncdump -h means headers only, no data

#ncview ${ddir}/${fname}${region_tag}_MJJASO.nc

In [5]:
# Examining the files

cdo -s timmean ${ddir}/${fname}${region_tag}_MJJASO.nc ${ddir}/${fname}${region_tag}_MJJASO_timmean.nc
#getting the mean precipitation for all time
#results in a 2D data set, space only (x,y)

cdo -s sub ${ddir}/${fname}${region_tag}_MJJASO.nc ${ddir}/${fname}${region_tag}_MJJASO_timmean.nc ${ddir}/${fname}${region_tag}_MJJASO_anom.nc
#subtracting the all time mean from each data set to find the anomaly
#recursive subtraction
#results in 3D data set, anomaly for all time (everyday) (x,y,t)

cdo -s yearmean ${ddir}/${fname}${region_tag}_MJJASO_anom.nc ${ddir}/${fname}${region_tag}_MJJASO_anom_yearmean.nc
#getting the mean anomaly per year
#reduces time data to just one per year instead of one per day
#results in 3D data set, anomaly per year (x,y,t)

ncview ${ddir}/${fname}${region_tag}_MJJASO_anom_yearmean.nc

Ncview 2.1.8 David W. Pierce  8 March 2017
http://meteora.ucsd.edu:80/~pierce/ncview_home_page.html
Copyright (C) 1993 through 2015, David W. Pierce
Ncview comes with ABSOLUTELY NO WARRANTY; for details type `ncview -w'.
This is free software licensed under the Gnu General Public License version 3; type `ncview -c' for redistribution details.

calculating min and maxes for precip...
X connection to :0 broken (explicit kill or server shutdown).


: 1

In [46]:
ncview -frames ${ddir}/${fname}${region_tag}_MJJASO_anom_yearmean.nc
ffmpeg -r 5 -i frame.%05d.png PHanomaly.gif
#-r 10 means framerate of 10
#-i means input
#%06d means 5 zeroes before the number 1, d is a placeholder for sequential numbers

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [47]:
#Spatial averaging

cdo -s fldmean ${ddir}/${fname}${region_tag}_MJJASO_anom_yearmean.nc ${ddir}/${fname}${region_tag}_MJJASO_anom_meanindex.nc
#getting the weighted average over all space (gridded data)
#results in 1D data, just anomaly of the whole Philippines over time
#note that axis will say precipitation rather than anomaly, since metadata remains unchanged

ncview ${ddir}/${fname}${region_tag}_MJJASO_anom_meanindex.nc
#download the .ps of the image pls
#filename 1996_2021_PH_Anomaly.ps

Ncview 2.1.8 David W. Pierce  8 March 2017
http://meteora.ucsd.edu:80/~pierce/ncview_home_page.html
Copyright (C) 1993 through 2015, David W. Pierce
Ncview comes with ABSOLUTELY NO WARRANTY; for details type `ncview -w'.
This is free software licensed under the Gnu General Public License version 3; type `ncview -c' for redistribution details.



In [3]:
gs -dSAFER -dBATCH -dNOPAUSE -sDEVICE=png16m -r300 -sOutputFile=1996_2021_PH_Anomaly.png 1996_2021_PH_Anomaly.ps

GPL Ghostscript 10.02.1 (2023-11-01)
Copyright (C) 2023 Artifex Software, Inc.  All rights reserved.
This software is supplied under the GNU AGPLv3 and comes with NO WARRANTY:
see the file COPYING for details.
Loading NimbusRoman-Regular font from /usr/share/ghostscript/10.02.1/Resource/Font/NimbusRoman-Regular... 3704328 2242592 1767648 468984 1 done.
Loading NimbusSans-Regular font from /usr/share/ghostscript/10.02.1/Resource/Font/NimbusSans-Regular... 3784344 2430217 1787848 478545 1 done.


In [48]:
#wet area index

# first we set to 1 all points that are with a positive anomaly
cdo -s gec,0 ${ddir}/${fname}${region_tag}_MJJASO_anom_yearmean.nc ${ddir}/${fname}${region_tag}_MJJASO_anom_binary.nc
#3D data set, anomaly per year (x,y,t), except 1 if positive and 0 if negative

# and now we can add up all the 1s to see what the wet area is
cdo -s fldmean ${ddir}/${fname}${region_tag}_MJJASO_anom_binary.nc ${ddir}/${fname}${region_tag}_MJJASO_wetarea_index.nc
#fldmean gives a weighted average over the Earth
#the number of positive points divided by the total number of points gives the fractional of positive anomaly
#AKA, the wet area index
#resulting data should be 1D, since we are averaging over space

ncview ${ddir}/${fname}${region_tag}_MJJASO_wetarea_index.nc
#download the .ps of the image pls
#filename 1996_2021_PH_WetIndex.ps

Ncview 2.1.8 David W. Pierce  8 March 2017
http://meteora.ucsd.edu:80/~pierce/ncview_home_page.html
Copyright (C) 1993 through 2015, David W. Pierce
Ncview comes with ABSOLUTELY NO WARRANTY; for details type `ncview -w'.
This is free software licensed under the Gnu General Public License version 3; type `ncview -c' for redistribution details.

X connection to :0 broken (explicit kill or server shutdown).


: 1

In [4]:
gs -dSAFER -dBATCH -dNOPAUSE -sDEVICE=png16m -r300 -sOutputFile=1996_2021_PH_WetIndex.png 1996_2021_PH_WetIndex.ps

GPL Ghostscript 10.02.1 (2023-11-01)
Copyright (C) 2023 Artifex Software, Inc.  All rights reserved.
This software is supplied under the GNU AGPLv3 and comes with NO WARRANTY:
see the file COPYING for details.
Loading NimbusRoman-Regular font from /usr/share/ghostscript/10.02.1/Resource/Font/NimbusRoman-Regular... 3704328 2242594 1767648 468987 1 done.
Loading NimbusSans-Regular font from /usr/share/ghostscript/10.02.1/Resource/Font/NimbusSans-Regular... 3784344 2430219 1787848 478717 1 done.


In [49]:
#number of extreme rainy days using 95th percentile

# make a index for extremes P95 for example
ifile=${ddir}/${fname}${region_tag}_MJJASO.nc
#shortcut to make line shorter

percen=95

cdo -s timpctl,${percen} $ifile -timmin $ifile -timmax $ifile ${ddir}/${fname}${region_tag}_MJJASO_p${percen}.nc
#compute 95th percentile over time of the file from timmin of 1996 to timmax of 2021, save to new file
#creates 2D data (x,y), since percentile is over time

cdo -s ge $ifile ${ddir}/${fname}${region_tag}_MJJASO_p${percen}.nc ${ddir}/${fname}${region_tag}_MJJASO_p${percen}_binary.nc
#if greater than the 95th percentile over all time, then save 1. Otherwise, 0.
#result is still 3D data (x,y,t)

# number of extreme rainy days per year
cdo -s yearsum ${ddir}/${fname}${region_tag}_MJJASO_p${percen}_binary.nc ${ddir}/${fname}${region_tag}_MJJASO_p${percen}_nevents.nc
#yearsum still gives 3D data, but lessens the time range from every day to every year.

ncview ${ddir}/${fname}${region_tag}_MJJASO_p${percen}_nevents.nc

HDF5-DIAG: Error detected in HDF5 (1.10.10) thread 1:
  #000: ../../../src/H5A.c line 484 in H5Aopen_by_name(): can't open attribute
    major: Attribute
    minor: Can't open object
  #001: ../../../src/H5Aint.c line 542 in H5A__open_by_name(): unable to load attribute info from object header
    major: Attribute
    minor: Unable to initialize object
  #002: ../../../src/H5Oattribute.c line 478 in H5O__attr_open_by_name(): can't open attribute
    major: Attribute
    minor: Can't open object
  #003: ../../../src/H5Adense.c line 397 in H5A__dense_open(): can't locate attribute in name index
    major: Attribute
    minor: Object not found
HDF5-DIAG: Error detected in HDF5 (1.10.10) thread 1:
  #000: ../../../src/H5A.c line 484 in H5Aopen_by_name(): can't open attribute
    major: Attribute
    minor: Can't open object
  #001: ../../../src/H5Aint.c line 542 in H5A__open_by_name(): unable to load attribute info from object header
    major: Attribute
    minor: Unable to initialize obj

: 1

In [50]:
ncview -frames ${ddir}/${fname}${region_tag}_MJJASO_p${percen}_nevents.nc
ffmpeg -r 5 -i frame.%05d.png PHp95.gif

Ncview 2.1.8 David W. Pierce  8 March 2017
http://meteora.ucsd.edu:80/~pierce/ncview_home_page.html
Copyright (C) 1993 through 2015, David W. Pierce
Ncview comes with ABSOLUTELY NO WARRANTY; for details type `ncview -w'.
This is free software licensed under the Gnu General Public License version 3; type `ncview -c' for redistribution details.

calculating min and maxes for precip...
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-li